# 01c — Bulk tier loader (`raw/Primary` + `raw/Secondary`)

**Phase 1c** (plan §6 Stage 1; §3.1 `scripts/bulk_ingest.py` lane). Runs immediately after `01_ingestion.ipynb` and `01b_language_detection_native.ipynb`.

Goal: idempotently walk the whole corpus and produce the same v2.1 spine that `01_ingestion` produced for three fixtures, but for **every** PDF / EPUB / `.packed/` tree under `raw/`. Then run Phase-1b language detection over the freshly created native pages in a single batched pass.

Pipeline:

`raw/{Primary,Secondary}/** -> discover_corpus -> bulk_ingest -> (TOPIC -> DOCUMENT -> CHAPTER -> SECTION -> PAGE + MinIO) -> detect_pages`

Concretely:

1. **Discovery** — `apps.backend.pipeline.bulk_loader.discover_corpus(...)` walks `raw/Primary/` and `raw/Secondary/`, deterministically picks every `*.pdf`, `*.epub`, and `*.packed/` directory, computes `tier`, `backend`, and the same path-hashed `document_id` that `ingest_path` would use, and reports anything it had to skip (`*.djvu`, `*.md` annotations, etc.).
2. **Idempotent ingest** — `bulk_ingest(...)` checks each `document_id` against `(:DOCUMENT)` in Neo4j and skips when already present (`status='skipped'`), so a partial run can be resumed by re-executing the cell. When `dry_run=True`, every fresh entry is reported with `status='would-ingest'` and zero side effects.
3. **LLM enrichment ON by default** — per AGENTS.md §11 entry dated 2026-05-17 ("Bulk ingest (01c) defaults to LLM-on"), every document gets one `deepseek-chat` call to refine the regex-extracted edition metadata; cost ≈ 800 chars × N docs.
4. **Phase-1b sweep** — after the ingest loop finishes, `bulk_ingest` calls `apps.backend.pipeline.lang_detect.detect_pages(driver)` once, classifying every brand-new native page (with `recompute_existing=False`, so already-classified pages are not retouched).

**Inputs**

- `notebooks/_artifacts/00_setup_smoke_test/health.json` — must report all-healthy.
- `notebooks/_artifacts/01_ingestion/ingestion.json` — proof Phase-1 plumbing works end-to-end on the smoke fixtures.
- `notebooks/_artifacts/01b_language_detection_native/lang_detect.json` — proof Phase-1b classifier works.
- A populated `raw/Primary/` and `raw/Secondary/` (post-`scripts/rename_corpus.py --apply`).

**Outputs**

- `notebooks/_artifacts/01c_tier_loader/inventory.json` — full discovery + per-doc ingest report, sortable by tier and backend.
- Neo4j: every corpus unit upserted into the `(:TOPIC)-[:CONTAIN]->(:DOCUMENT)-[:CONSIST_OF]->(:CHAPTER)-[:INCLUDE]->(:SECTION)-[:INCLUDE]->(:PAGE)` spine.
- MinIO `ancient-pages/`: page images for every OCR-bound page across the corpus.

**Two-mode design**

- **Smoke mode** (default in cells below): `MAX_DOCS=3`, `MAX_PAGES_PER_DOC=20`, `RUN_LANG_DETECT=True`. Completes in a couple of minutes, sanity-checks the discovery + idempotency path without burning Silra credits or MinIO storage.
- **Full mode** (opt-in): flip `RUN_FULL=True` in cell `full-run`. Removes the doc and page caps so the entire corpus is ingested. Expect 30-60 minutes wall-clock dominated by the larger primaries (`册府元龟` alone is ~870 MB / 1000+ pages).

**Next**: `02_normalization.ipynb` (philological normalization smoke + reverse-index).

In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import json
import logging
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'pyproject.toml').exists(), f'cannot locate repo root from {Path.cwd()}'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

loaded = load_dotenv(REPO_ROOT / '.env')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
logging.getLogger('apps.backend.pipeline.ingest').setLevel(logging.WARNING)
logging.getLogger('neo4j.notifications').setLevel(logging.WARNING)

ARTIFACT_DIR = REPO_ROOT / 'notebooks' / '_artifacts' / '01c_tier_loader'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RAW_ROOT       = REPO_ROOT / 'raw'
PRIOR_HEALTH   = REPO_ROOT / 'notebooks' / '_artifacts' / '00_setup_smoke_test' / 'health.json'
PRIOR_INGEST   = REPO_ROOT / 'notebooks' / '_artifacts' / '01_ingestion' / 'ingestion.json'
PRIOR_LANG     = REPO_ROOT / 'notebooks' / '_artifacts' / '01b_language_detection_native' / 'lang_detect.json'

print(f'repo root      : {REPO_ROOT}')
print(f'.env loaded    : {loaded}')
print(f'artifact dir   : {ARTIFACT_DIR}')
print(f'raw root       : {RAW_ROOT}  (exists: {RAW_ROOT.exists()})')
print(f'NEO4J_URI      : {os.getenv("NEO4J_URI", "<unset>")}')
print(f'CHAT_LLM_MODEL : {os.getenv("CHAT_LLM_MODEL", "<unset>")}')

repo root      : /Users/mohasani/Ancient
.env loaded    : True
artifact dir   : /Users/mohasani/Ancient/notebooks/_artifacts/01c_tier_loader
raw root       : /Users/mohasani/Ancient/raw  (exists: True)
NEO4J_URI      : bolt://localhost:7687
CHAT_LLM_MODEL : deepseek-chat


## 1. Refuse to advance if Phase 0 / 1 / 1b are red

Bulk ingest is the most expensive notebook in the chain (LLM calls + MinIO uploads + Neo4j writes for the whole corpus). Bail out **before** any of that if the upstream artifacts are missing or unhealthy — there's no point spending Silra credits to chase a config bug.

In [2]:
assert PRIOR_HEALTH.exists(),  f'missing {PRIOR_HEALTH}: run 00_setup_smoke_test.ipynb first'
assert PRIOR_INGEST.exists(),  f'missing {PRIOR_INGEST}: run 01_ingestion.ipynb first'
assert PRIOR_LANG.exists(),    f'missing {PRIOR_LANG}: run 01b_language_detection_native.ipynb first'
assert RAW_ROOT.exists(),      f'missing corpus root: {RAW_ROOT}'

health   = json.loads(PRIOR_HEALTH.read_text())
ingest_a = json.loads(PRIOR_INGEST.read_text())
lang_a   = json.loads(PRIOR_LANG.read_text())

for svc in ('silra', 'neo4j', 'minio'):
    ok = bool(health.get(svc, {}).get('ok'))
    print(f'  {svc:6s}: {"OK" if ok else "FAIL"}')
    assert ok, f'service {svc} unhealthy in Phase 0 artifact: {health.get(svc)}'

prior_docs = ingest_a.get('documents') or ingest_a.get('reports') or []
print(f'  prior 01    : {len(prior_docs)} docs ingested')
assert prior_docs, 'Phase 1 ingested zero documents; rerun 01_ingestion.ipynb'

ld_processed = lang_a.get('pages_processed', 0)
print(f'  prior 01b   : {ld_processed} native pages classified')

  silra : OK
  neo4j : OK
  minio : OK
  prior 01    : 3 docs ingested
  prior 01b   : 0 native pages classified


## 2. Wire clients + reaffirm schema

We open the Neo4j driver and MinIO client and re-run `init_schema(driver)` so the spine constraints / indexes are guaranteed before any bulk MERGE. We do **not** call `migrate_labels_to_uppercase` here — that's a one-time database surgery; if you need it, run `01_ingestion.ipynb` first which has the explicit migrate cell.

In [3]:
from apps.backend.graph.neo4j_client import get_driver, ping as neo4j_ping
from apps.backend.graph.schema import init_schema
from apps.backend.storage.minio_client import get_minio_client, ensure_bucket
from apps.backend.llm.silra import ping as silra_ping

silra_status = silra_ping()
print('silra ping :', 'OK' if silra_status.get('ok') else 'FAIL')
assert silra_status.get('ok'), silra_status

driver = get_driver()
neo_status = neo4j_ping(driver=driver)
print('neo4j ping :', 'OK' if neo_status.get('ok') else 'FAIL', '-', neo_status.get('server_version'))
assert neo_status.get('ok'), neo_status

bucket = os.getenv('MINIO_BUCKET_PAGES', 'ancient-pages')
minio_client = get_minio_client()
ensure_bucket(minio_client, bucket)
print(f"minio bucket '{bucket}' ready")

init_report = init_schema(driver=driver)
print(f"schema: {len(init_report.get('constraints', []))} constraints, "
      f"{len(init_report.get('vector_indexes', []))} vector indexes, "
      f"{len(init_report.get('lookup_indexes', []))} lookup indexes")
assert not init_report.get('errors'), init_report['errors']

2026-05-19 00:20:50,397 INFO httpx: HTTP Request: POST https://api.silra.cn/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19 00:20:50,402 INFO apps.backend.llm.silra: silra.chat model=deepseek-chat prompt=162 completion=1 total=163
2026-05-19 00:20:51,606 INFO httpx: HTTP Request: POST https://api.silra.cn/v1/embeddings "HTTP/1.1 200 OK"
2026-05-19 00:20:51,612 INFO apps.backend.llm.silra: silra.embed model=text-embedding-v4 prompt=2 total=2


silra ping : OK
neo4j ping : OK - 5.18.1
minio bucket 'ancient-pages' ready
schema: 16 constraints, 5 vector indexes, 22 lookup indexes


## 3. Discovery preview (read-only)

Walk `raw/` and enumerate every corpus unit **before** touching Neo4j. The preview shows:

- per-tier counts and total size on disk;
- per-backend counts (epub / pdf / packed_md);
- the full skip list with reasons (so `.djvu` and stray `.md` annotations show up explicitly).

Setting `count_pages=True` opens every PDF/EPUB to read `page_count` for the discovery table. It's cheap (~10s for the whole corpus) and worth it for the inventory.

In [4]:
from apps.backend.pipeline.bulk_loader import discover_corpus

entries, skipped = discover_corpus(
    RAW_ROOT,
    include_primary=True,
    include_secondary=True,
    count_pages=True,
    repo_root=REPO_ROOT,
)

from collections import Counter

tier_counts    = Counter(e.tier    for e in entries)
backend_counts = Counter(e.backend for e in entries)
tier_bytes     = Counter()
tier_pages     = Counter()
for e in entries:
    tier_bytes[e.tier]  += e.size_bytes
    if e.page_count_hint is not None:
        tier_pages[e.tier] += e.page_count_hint

def _fmt_bytes(n: int) -> str:
    for unit in ('B', 'KB', 'MB', 'GB'):
        if n < 1024:
            return f'{n:.1f} {unit}'
        n /= 1024
    return f'{n:.1f} TB'

print(f'discovered: {len(entries)} entries, {len(skipped)} skipped\n')
print('  by tier:')
for t, n in sorted(tier_counts.items()):
    print(f'    {t:10s} {n:4d} entries  {_fmt_bytes(tier_bytes[t]):>10s}  {tier_pages[t]:>6d} pages (hinted)')
print('  by backend:')
for b, n in sorted(backend_counts.items()):
    print(f'    {b:10s} {n:4d}')

if skipped:
    print(f'\nskipped ({len(skipped)}):')
    reason_counts = Counter(s.reason for s in skipped)
    for reason, n in reason_counts.most_common():
        print(f'  {n:3d}  {reason}')

discovered: 74 entries, 5 skipped

  by tier:
    primary      15 entries      1.4 GB   27193 pages (hinted)
    secondary    59 entries    810.3 MB    8300 pages (hinted)
  by backend:
    epub          7
    packed_md     1
    pdf          66

skipped (5):
    4  markdown annotation file, not a corpus unit
    1  .djvu has no reader yet (deferred to Phase 3 OCR)


### 3a. Per-entry preview (first 20)

Quick eyeball that the path -> tier -> backend -> document_id mapping looks right. Anything weird here (e.g. a primary book resolving to `secondary` because of a path quirk) blocks the run.

In [5]:
for e in entries[:20]:
    pages = e.page_count_hint if e.page_count_hint is not None else '?'
    print(f'  [{e.tier:9s}] [{e.backend:9s}] {e.document_id:30s}  pages={str(pages):>5s}  {e.relative_path}')
if len(entries) > 20:
    print(f'  ... and {len(entries) - 20} more')

  [primary  ] [pdf      ] 册府元龟__edcb503758                pages=13234  raw/Primary/册府元龟.pdf
  [primary  ] [pdf      ] 唐令拾遗__1b265b6f5b                pages=  961  raw/Primary/唐令拾遗.pdf
  [primary  ] [pdf      ] 唐令拾遗补__264d0c87d2               pages=  763  raw/Primary/唐令拾遗补.pdf
  [primary  ] [pdf      ] 唐大诏令集__5be881cf9b               pages=  716  raw/Primary/唐大诏令集.pdf
  [primary  ] [pdf      ] 唐大诏令集补编_上__acad10c8c3           pages=  897  raw/Primary/唐大诏令集补编 上.pdf
  [primary  ] [pdf      ] 唐大诏令集补编_下__9093181fac           pages= 1027  raw/Primary/唐大诏令集补编 下.pdf
  [primary  ] [pdf      ] 唐律疏議箋解__3fbb3392d0              pages= 2278  raw/Primary/唐律疏議箋解.pdf
  [primary  ] [epub     ] 唐摭言__61690c92f7                 pages=   17  raw/Primary/唐摭言.epub
  [primary  ] [pdf      ] 唐會要__7f967361d2                 pages=  845  raw/Primary/唐會要.pdf
  [primary  ] [epub     ] 新五代史__aa61571e9c                pages=   77  raw/Primary/新五代史.epub
  [primary  ] [epub     ] 新唐书__ce94b87d96                 pages=  

## 4. Dry-run: planned writes (zero side effects)

Now ask `bulk_ingest(..., dry_run=True)` what it **would** do. Already-ingested docs are reported as `status='skipped'`; fresh docs are `status='would-ingest'`. Use this to confirm that the resume path is wired correctly (the 3 docs from `01_ingestion.ipynb` must show up as skipped).

In [6]:
from apps.backend.pipeline.bulk_loader import bulk_ingest

dry = bulk_ingest(
    driver,
    entries,
    dry_run=True,
    skip_existing=True,
)

print(f'dry-run summary  (took {dry.duration_seconds:.2f}s)')
print(f'  discovered      : {dry.discovered}')
print(f'  skipped_existing: {dry.skipped_existing}')
print(f'  would_ingest    : {dry.would_ingest}')
print(f'  by_tier         : {dict(dry.by_tier)}')
print(f'  by_backend      : {dict(dry.by_backend)}')

fresh_preview = [d for d in dry.documents if d.status == 'would-ingest'][:10]
if fresh_preview:
    print('\nfirst 10 docs that WOULD be ingested:')
    for d in fresh_preview:
        print(f'  [{d.tier:9s}] [{d.backend:9s}] {d.relative_path}')
skipped_preview = [d for d in dry.documents if d.status == 'skipped'][:10]
if skipped_preview:
    print('\nfirst 10 docs already in Neo4j (will be skipped):')
    for d in skipped_preview:
        print(f'  pages={d.pages_total:>5d}  [{d.tier:9s}] [{d.backend:9s}] {d.relative_path}')

dry-run summary  (took 0.00s)
  discovered      : 74
  skipped_existing: 0
  would_ingest    : 74
  by_tier         : {'primary': 15, 'secondary': 59}
  by_backend      : {'pdf': 66, 'epub': 7, 'packed_md': 1}

first 10 docs that WOULD be ingested:
  [primary  ] [pdf      ] raw/Primary/册府元龟.pdf
  [primary  ] [pdf      ] raw/Primary/唐令拾遗.pdf
  [primary  ] [pdf      ] raw/Primary/唐令拾遗补.pdf
  [primary  ] [pdf      ] raw/Primary/唐大诏令集.pdf
  [primary  ] [pdf      ] raw/Primary/唐大诏令集补编 上.pdf
  [primary  ] [pdf      ] raw/Primary/唐大诏令集补编 下.pdf
  [primary  ] [pdf      ] raw/Primary/唐律疏議箋解.pdf
  [primary  ] [epub     ] raw/Primary/唐摭言.epub
  [primary  ] [pdf      ] raw/Primary/唐會要.pdf
  [primary  ] [epub     ] raw/Primary/新五代史.epub


## 5. Smoke run (3 fresh docs, capped pages)

Default run: ingest at most three fresh documents, capping each at 20 pages. This is the equivalent of `01_ingestion.ipynb` but driven through the bulk loader, so we get the same end-to-end coverage with no manual fixture wiring. Already-ingested docs are skipped, so this cell is **safe to re-run** — it will advance through the corpus three docs at a time.

After the ingest loop, `bulk_ingest` calls `detect_pages(driver, recompute_existing=False)` once to classify every brand-new native page in a single pass.

In [7]:
MAX_DOCS          = 3
MAX_PAGES_PER_DOC = 20
RUN_LANG_DETECT   = True
ENRICH_WITH_LLM   = True

def _on_start(idx, total, entry):
    pages = entry.page_count_hint if entry.page_count_hint is not None else '?'
    print(f'  [{idx}/{total}] -> {entry.relative_path}  ({entry.backend}, ~{pages} pages, tier={entry.tier})', flush=True)

def _on_done(idx, total, result):
    tag = 'OK ' if result.status == 'ingested' else 'ERR'
    print(
        f'  [{idx}/{total}] {tag} status={result.status}  '
        f'pages={result.pages_total} (native={result.pages_native}, ocr={result.pages_ocr})  '
        f'chapters={result.chapters_written} sections={result.sections_written}  '
        f'minio={result.pages_uploaded_to_minio}  '
        f'{result.duration_seconds:.2f}s'
        + (f'  ERROR={result.error}' if result.error else ''),
        flush=True,
    )

smoke = bulk_ingest(
    driver,
    entries,
    minio_client=minio_client,
    bucket=bucket,
    repo_root=REPO_ROOT,
    skip_existing=True,
    enrich_metadata_with_llm=ENRICH_WITH_LLM,
    max_pages_per_doc=MAX_PAGES_PER_DOC,
    max_docs=MAX_DOCS,
    run_lang_detect=RUN_LANG_DETECT,
    on_doc_start=_on_start,
    on_doc_done=_on_done,
)

print('\nsmoke-run summary')
print(f'  ingested            : {smoke.ingested}')
print(f'  skipped_existing    : {smoke.skipped_existing}')
print(f'  failed              : {smoke.failed}')
print(f'  pages_total         : {smoke.pages_total}')
print(f'  pages_uploaded_minio: {smoke.pages_uploaded_to_minio}')
print(f'  lang_detect (processed/written): {smoke.lang_detect_pages_processed}/{smoke.lang_detect_pages_written}')
print(f'  duration_seconds    : {smoke.duration_seconds:.2f}')

for d in smoke.documents:
    if d.status == 'failed':
        print(f'  FAILED  {d.relative_path}  -> {d.error}')


smoke-run summary
  ingested            : 0
  skipped_existing    : 74
  failed              : 0
  pages_total         : 0
  pages_uploaded_minio: 0
  lang_detect (processed/written): 0/0
  duration_seconds    : 0.12


## 6. Full run (opt-in)

Flip `RUN_FULL = True` below to ingest **every** remaining doc. Expect 30-60 minutes — dominated by the big primaries (册府元龟, 通典, 旧唐书, 新唐书). Each run is idempotent and resumable: kill it any time, re-execute, and it picks up where it left off.

Tips:

- Keep `max_pages_per_doc=None` so structure extraction sees the full TOC; the per-doc cap from the smoke run leaves the v2.1 spine truncated for big books.
- Set `enrich_metadata_with_llm=False` if you want to skip LLM calls during a fast re-run.
- Watch `docker stats` and `minio` disk usage — primary EPUBs upload thousands of page images.

In [8]:
RUN_FULL = False  # flip to True for the full corpus pass

if RUN_FULL:
    full = bulk_ingest(
        driver,
        entries,
        minio_client=minio_client,
        bucket=bucket,
        repo_root=REPO_ROOT,
        skip_existing=True,
        enrich_metadata_with_llm=True,
        max_pages_per_doc=None,
        max_docs=None,
        run_lang_detect=True,
        on_doc_start=_on_start,
        on_doc_done=_on_done,
    )
    print('\nfull-run summary')
    print(f'  ingested            : {full.ingested}')
    print(f'  skipped_existing    : {full.skipped_existing}')
    print(f'  failed              : {full.failed}')
    print(f'  pages_total         : {full.pages_total}')
    print(f'  pages_uploaded_minio: {full.pages_uploaded_to_minio}')
    print(f'  lang_detect         : {full.lang_detect_pages_written}/{full.lang_detect_pages_processed}')
    print(f'  duration_seconds    : {full.duration_seconds:.2f}')
else:
    print('RUN_FULL is False — skipping full corpus pass. Use the smoke cell above for incremental progress.')
    full = None

RUN_FULL is False — skipping full corpus pass. Use the smoke cell above for incremental progress.


## 7. Verify Neo4j state

Independent of `BulkLoadReport`, query Neo4j directly so we know what's actually in the graph after this notebook's writes.

In [9]:
from apps.backend.pipeline.lang_detect import language_summary

Q_DOC_COUNTS = '''
MATCH (d:DOCUMENT)
OPTIONAL MATCH (d)-[:CONSIST_OF]->(:CHAPTER)-[:INCLUDE]->(:SECTION)-[:INCLUDE]->(p:PAGE)
RETURN d.tier AS tier, count(DISTINCT d) AS docs, count(DISTINCT p) AS pages
ORDER BY tier
'''
Q_TOTAL = '''
MATCH (d:DOCUMENT)
OPTIONAL MATCH (d)-[:CONSIST_OF]->(c:CHAPTER)
OPTIONAL MATCH (c)-[:INCLUDE]->(s:SECTION)
OPTIONAL MATCH (s)-[:INCLUDE]->(p:PAGE)
RETURN count(DISTINCT d) AS docs,
       count(DISTINCT c) AS chapters,
       count(DISTINCT s) AS sections,
       count(DISTINCT p) AS pages
'''

with driver.session() as session:
    per_tier = list(session.run(Q_DOC_COUNTS))
    totals   = session.run(Q_TOTAL).single().data()

print('Neo4j state (whole graph):')
print(f'  documents: {totals["docs"]}')
print(f'  chapters : {totals["chapters"]}')
print(f'  sections : {totals["sections"]}')
print(f'  pages    : {totals["pages"]}')
print('\nby tier:')
for row in per_tier:
    print(f'  tier={row["tier"]:>10s}  docs={row["docs"]:>4d}  pages={row["pages"]:>6d}')

lang_summary = language_summary(driver)
print('\nlanguage rollup:')
by_lang = lang_summary.get('by_language', {}) if isinstance(lang_summary, dict) else {}
for lang, n in sorted(by_lang.items(), key=lambda kv: -kv[1]):
    print(f'  {lang:14s} {n:>6d}')

Neo4j state (whole graph):
  documents: 74
  chapters : 3246
  sections : 4378
  pages    : 18272

by tier:
  tier=   primary  docs=  15  pages=  6472
  tier= secondary  docs=  59  pages= 11800

language rollup:
  zh-classical     9981
  (unset)          6524
  zh-modern        1572
  unknown           167
  mixed              27
  ja                  1


## 8. Persist `inventory.json`

The artifact carries the full discovery (entries + skipped), the smoke-run report, and the optional full-run report. Phase 2 (`02_normalization.ipynb`) will not consume this directly, but the per-doc page counts make it the canonical source for the corpus dashboard.

In [10]:
artifact = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'repo_root': str(REPO_ROOT),
    'raw_root': str(RAW_ROOT),
    'discovery': {
        'entries': [e.to_dict() for e in entries],
        'skipped': [s.to_dict() for s in skipped],
        'by_tier': dict(tier_counts),
        'by_backend': dict(backend_counts),
        'total_bytes': sum(e.size_bytes for e in entries),
        'total_pages_hinted': sum((e.page_count_hint or 0) for e in entries),
    },
    'dry_run': dry.to_dict(),
    'smoke_run': smoke.to_dict(),
    'full_run': full.to_dict() if full is not None else None,
    'neo4j': {
        'totals': totals,
        'per_tier': [dict(row) for row in per_tier],
        'language_rollup': lang_summary,
    },
}

out_path = ARTIFACT_DIR / 'inventory.json'
out_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False))
print(f'wrote {out_path}  ({out_path.stat().st_size / 1024:.1f} KB)')

wrote /Users/mohasani/Ancient/notebooks/_artifacts/01c_tier_loader/inventory.json  (110.8 KB)


## 9. Wrap up

We deliberately do **not** call `driver.close()` here — that would invalidate the `driver` binding and break re-execution of any cell above. The kernel-shutdown handler closes the connection pool automatically when you close the notebook.


In [11]:
print('01c done. driver intentionally left open so verify/persist cells stay re-runnable;')
print('  the kernel shutdown handler will release the connection pool.')
print('  if you have already executed `driver.close()` by hand, re-run the `clients` cell above.')

01c done. driver intentionally left open so verify/persist cells stay re-runnable;
  the kernel shutdown handler will release the connection pool.
  if you have already executed `driver.close()` by hand, re-run the `clients` cell above.
